# Direction 2 · Phase C · Step 7c-1: Waterbirds One-Shot Test Confirmation (`ResNet-50` + `ViT-B/16`)

**Settings:** Enable **GPU T4 x2** and **Internet ON**.

**Zero manual uploads required & zero test-split tuning:**
- Clones `spurious_correlation_mechanisms` from GitHub.
- Extracts Waterbirds (~490 MB) into `/tmp/step7c_wb_assets` and auto-recovers all 5 `standard_erm` + 2 `erm_star` ResNet-50 checkpoints plus all 3 `ViT-B/16` checkpoints (`noerula/rep-repair-step7a-vit-kaggle`).
- Calibrates every checkpoint, neutral baseline, and scalar decision threshold strictly on validation (`split == 1`), then evaluates **once** across `split == 2` (`5,794` held-out test rows) in parallel across `2x T4` GPUs using frozen **`pre_l4_repair_w005`** (and its ViT cross-architecture transfer **`vit_pre_b6_repair_w005`**).
- Outputs `/kaggle/working/rep_repair_step7c_waterbirds_test.zip`.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
import torch

REPO_URL = 'https://github.com/chautanphat/waterbirds-shortcut-mitigation.git'
BRANCH = 'spurious_correlation_mechanisms'
CODE_DIR = Path('/kaggle/working/repo')
WORK_DIR = Path('/tmp/step7c_wb_assets')
OUT_DIR = Path('/kaggle/working/rep_repair_step7c_waterbirds_test')

n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'CUDA: {torch.cuda.is_available()} | GPUs ({n_gpus}): {[torch.cuda.get_device_name(i) for i in range(n_gpus)]}')

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CODE_DIR)], check=True)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

subprocess.run([sys.executable, '-B', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_rep_repair_step7c.py', '-v'], check=True)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cmd = [
    sys.executable, '-B', '-m', 'scripts.analysis.rep_repair_step7c_waterbirds_test',
    '--work-dir', str(WORK_DIR),
    '--output', str(OUT_DIR),
    '--device', device,
    '--max-gpus', '2',
    '--workers', '0',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown((OUT_DIR / 'README.md').read_text(encoding='utf-8')))
display(Image(filename=str(OUT_DIR / 'step7c_waterbirds_test.png')))

zip_path = Path('/kaggle/working/rep_repair_step7c_waterbirds_test.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.rglob('*')):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT_DIR))
print(f'Saved results bundle: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')